# Reunión 17 — Especificidad, matriz de confusión y barrido de combinaciones

Continúa `reunion16`. Reutiliza **exactamente** su pipeline (derivación de variables, partición
train/test, imputación/escalado solo en train y selección 3 métodos → ≥2/3) y añade lo pedido en la
reunión:

1. **Especificidad** (= TN/(TN+FP), tasa de verdaderos negativos) y **matriz de confusión** en la tabla
   de clasificadores de cada dataset.
2. La **matriz de confusión para train y para test**.
3. Con las **variables seleccionadas** de cada dataset (las de `reunion16`), un **barrido de
   combinaciones** (estilo clustering) para ver qué combinación **maximiza el F1**. Para las
   combinaciones con **F1 > 0,55**, tabla de métricas (F1, precisión, recall, ROC, especificidad,
   matriz de confusión).
4. **GridSearch** sobre las combinaciones con F1 > 0,55: (a) optimizando **F1** y (b) optimizando
   **especificidad**.
5. **Tablas comparativas** del punto 3 (sin GridSearch) frente al punto 4 (con GridSearch).

> **Decisiones (para poder defenderlas):**
> - **Modelo del barrido y del GridSearch = Regresión Logística (`class_weight='balanced'`).** Igual que
>   en clustering se fija UN algoritmo y se barren combinaciones de variables, aquí fijamos uno. La
>   LogReg es rápida (hay que evaluar miles de combinaciones), da probabilidades (ROC), admite el peso de
>   clase y permite una comparación limpia *sin GridSearch* vs *con GridSearch*. Los 6 clasificadores
>   siguen en el punto 1.
> - **El umbral F1 > 0,55 se mide por validación cruzada en train** (no en test), respetando la norma de
>   la tutora: *todo sobre train; el test solo para la predicción final*. En las tablas se muestran
>   igualmente las métricas de test.
> - **Aviso honesto:** optimizar *solo* especificidad empuja al clasificador trivial "todo negativo"
>   (especificidad≈1 pero recall≈0). Se reporta junto a F1/recall para ver el compromiso.
> - Para que el barrido sea abordable, se acota el tamaño máximo de combinación a un presupuesto
>   (~5.000 combinaciones/dataset); se documenta el tamaño usado en cada caso.


In [1]:
import os, warnings
os.environ['PYTHONWARNINGS'] = 'ignore'   # lo heredan los workers de GridSearch/RFECV (n_jobs=-1)
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
np.seterr(all='ignore')

RANDOM_STATE = 42

df  = pd.read_csv('datos.csv');               df['id']  = df['id'].astype(float)
df2 = pd.read_csv('datos_preprocesados.csv'); df2['id'] = df2['id'].astype(float)
print('datos.csv:', df.shape, '| datos_preprocesados.csv:', df2.shape)

datos.csv: (459, 344) | datos_preprocesados.csv: (608, 433)


## 1. Derivación de variables y outcome (idéntico a `reunion16`)

In [2]:
# Variables compuestas (definiciones EXACTAS de reunion10)
cols_ant_obstetrico = ['parto_previo_menor37_pre','aborto_menor20','ant_cir','ant_peg','ant_obito','ant_pe','ant_hellp']
cols_ant_medico     = ['ant_cesarea','ant_diabetes_pregest','hta_pregest','sindr_antifosfolipido','enf_autoinm','fuma','alcohol','drogas']
cols_hipertension   = ['pe','sd_hellp','hipertension_gest']
cols_placentaria    = ['cir','peg','desprendimiento_placenta']
cols_materna_grave  = ['uci_materna_ucoi','hemocerebral_ictus','trombosis_venosa_prof']
cols_metabolica     = ['diabetes_gest','colestasis_intrahepatica']

def _num(cols):
    return df[cols].apply(pd.to_numeric, errors='coerce')

df['ant_obstetrico']         = (_num(cols_ant_obstetrico).sum(axis=1) >= 1).astype(int)
df['ant_medico']             = (_num(cols_ant_medico).sum(axis=1) >= 1).astype(int)
df['trastorno_hipertensivo'] = _num(cols_hipertension).max(axis=1)
df['comp_placentaria']       = _num(cols_placentaria).max(axis=1)
df['comp_maternaGrave']      = _num(cols_materna_grave).max(axis=1)
df['comp_metabolica']        = _num(cols_metabolica).max(axis=1)

# Variables MoM (fórmula de reunion14)
mom_src = ['valor_sflt1','valor_plgf','ratio_sflt1_plgf','eg_deter_sflt1_plgf','creatinina']
df = df.merge(df2[['id'] + mom_src].drop_duplicates('id'), on='id', how='left')

def calcular_mom(d):
    d = d.copy()
    eg = d['eg_deter_sflt1_plgf']
    cond = [(eg>=10)&(eg<15),(eg>=15)&(eg<20),(eg>=20)&(eg<24),
            (eg>=24)&(eg<29),(eg>=29)&(eg<34),(eg>=34)&(eg<37),(eg>=37)]
    div_ratio = [24.8,10.5,4.92,3.06,3.75,9.03,19.6]
    div_sflt1 = [1328,1355,1299,1355,1742,2552,3485]
    div_plgf  = [52.6,135,264,465,471,284,191]
    d['ratio_MoM'] = d['ratio_sflt1_plgf'] / np.select(cond, div_ratio, default=np.nan)
    d['sflt1_MoM'] = d['valor_sflt1']      / np.select(cond, div_sflt1, default=np.nan)
    d['plgf_MoM']  = d['valor_plgf']       / np.select(cond, div_plgf,  default=np.nan)
    return d
df = calcular_mom(df)

# Codificaciones
df['etnia_caucasica']       = (df['etnia'] == 'Blanca').astype(int)
df['concepcion_espontanea'] = (df['concepcion'] == 'Espontanea').astype(int)
tomo_num = pd.to_numeric(df['tomo_durante_gest'], errors='coerce')
df['tomo'] = np.where(tomo_num.notna(), (tomo_num >= 1).astype(float), np.nan)
df['tabaco']    = df['fuma_post']
df['dieta']     = df['score_dieta']
df['ejercicio'] = df['act_fisica_total']
df['estres']    = df['score_estres']
cols_ant_med_act = ['hta_cronica','dislipemia','diabetes_mellitus_1_2','enf_renal_cronica','enf_autoinm_post','ant_infarto_ictus']
cols_ant_med_act = [c for c in cols_ant_med_act if c in df.columns]
df['ant_medico_actual'] = (df[cols_ant_med_act].apply(pd.to_numeric, errors='coerce').fillna(0).sum(axis=1) >= 1).astype(int)

# Outcome
outcome_vars = ['ta_sistolica','ta_diastolica','masa_vi_tdiast_indexada','fevi_simpson_4c','cardiopatia_estructural']
coh = df.dropna(subset=outcome_vars).copy()
coh['riesgo_cv_compuesto'] = (
    (coh['ta_sistolica'] >= 140) | (coh['ta_diastolica'] >= 90) |
    (coh['masa_vi_tdiast_indexada'] > 95) | (coh['fevi_simpson_4c'] < 50) |
    (coh['cardiopatia_estructural'] == 1)
).astype(int)
print(f'Cohorte: N={len(coh)}, positivos={int(coh.riesgo_cv_compuesto.sum())} ({coh.riesgo_cv_compuesto.mean()*100:.1f}%)')

Cohorte: N=369, positivos=57 (15.4%)


In [3]:
# Definición de los 4 datasets (listas de la tutora)
ds1_embarazo = ['peso_ini_gest','peso_fin_gest','aumento_peso_gest','talla','imc_ini_gest','peso_rn',
    'apgar_1min','apgar_5min','apgar_10min','eg_parto','sflt1_MoM','tas_1tri','tad_1tri','edad_materna_gest',
    'plgf_MoM','ratio_MoM','etnia_caucasica','concepcion_espontanea','tipo_parto','tomo','valor_plgf',
    'ratio_sflt1_plgf','ant_obstetrico','ant_medico','trastorno_hipertensivo','comp_placentaria',
    'comp_maternaGrave','comp_metabolica']
ds2_clasicas = ['edad_actual','peso_actual','imc_actual','ant_medico_actual','tabaco','dieta','ejercicio','estres']
analitica = ['grasa_visceral','ratio_cintura_cadera','hemoglobina_glicada','ldl','hdl','trigliceridos',
             'creatinina','urato_acidourico']
ds3_clasicas_analitica = ds2_clasicas + analitica
ds4_completo = ds1_embarazo + ds3_clasicas_analitica

DATASETS = {'1_embarazo': ds1_embarazo, '2_clasicas': ds2_clasicas,
            '3_clasicas_analitica': ds3_clasicas_analitica, '4_completo': ds4_completo}
NOMINALES = ['tipo_parto', 'tabaco']
for n, cols in DATASETS.items():
    print(f'Dataset {n}: {len(cols)} variables')

Dataset 1_embarazo: 28 variables
Dataset 2_clasicas: 8 variables
Dataset 3_clasicas_analitica: 16 variables
Dataset 4_completo: 44 variables


## 2. Pipeline y selección de variables (idéntico a `reunion16`)

In [4]:
from sklearn.model_selection import (train_test_split, StratifiedKFold, GridSearchCV,
                                     cross_val_score, cross_val_predict)
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer, SimpleImputer
from sklearn.ensemble import ExtraTreesRegressor, RandomForestClassifier
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.feature_selection import RFECV
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from xgboost import XGBClassifier
from sklearn.metrics import (f1_score, precision_score, recall_score, roc_auc_score,
                             confusion_matrix, make_scorer)
from collections import Counter
from itertools import combinations
from math import comb

def preparar(coh, feature_cols, nominales):
    cols = [c for c in feature_cols if c in coh.columns]
    nom  = [c for c in nominales if c in cols]
    num  = [c for c in cols if c not in nom]
    X = coh[cols].copy()
    X[num] = X[num].apply(pd.to_numeric, errors='coerce').astype(float)
    y = coh['riesgo_cv_compuesto'].values
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE)
    num = [c for c in num if Xtr[c].notna().any()]
    if num:
        imp = IterativeImputer(
            estimator=ExtraTreesRegressor(n_estimators=50, max_depth=4, min_samples_leaf=3, random_state=1234),
            random_state=1234, max_iter=20, imputation_order='ascending', initial_strategy='mean',
            min_value=[Xtr[c].min() for c in num], max_value=[Xtr[c].max() for c in num])
        tr_num = pd.DataFrame(imp.fit_transform(Xtr[num]), columns=num, index=Xtr.index)
        te_num = pd.DataFrame(imp.transform(Xte[num]),     columns=num, index=Xte.index)
    else:
        tr_num = pd.DataFrame(index=Xtr.index); te_num = pd.DataFrame(index=Xte.index)
    if nom:
        m = SimpleImputer(strategy='most_frequent')
        tr_c = pd.DataFrame(m.fit_transform(Xtr[nom]), columns=nom, index=Xtr.index)
        te_c = pd.DataFrame(m.transform(Xte[nom]),     columns=nom, index=Xte.index)
        ohe = OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore')
        tr_oh = ohe.fit_transform(tr_c); te_oh = ohe.transform(te_c)
        names = ohe.get_feature_names_out(nom)
        tr_c = pd.DataFrame(tr_oh, columns=names, index=Xtr.index)
        te_c = pd.DataFrame(te_oh, columns=names, index=Xte.index)
    else:
        tr_c = pd.DataFrame(index=Xtr.index); te_c = pd.DataFrame(index=Xte.index)
    tr = pd.concat([tr_num, tr_c], axis=1); te = pd.concat([te_num, te_c], axis=1)
    sc = StandardScaler()
    Xtr_s = pd.DataFrame(sc.fit_transform(tr), columns=tr.columns, index=tr.index)
    Xte_s = pd.DataFrame(sc.transform(te),     columns=te.columns, index=te.index)
    return Xtr_s, Xte_s, ytr, yte

def sel_lasso_stability(Xtr, ytr, n_iter=80, frac=0.5, C=0.5, umbral=0.6, seed=RANDOM_STATE):
    rng = np.random.RandomState(seed)
    n = len(ytr); m = int(n*frac); cuenta = np.zeros(Xtr.shape[1])
    for _ in range(n_iter):
        idx = rng.choice(n, m, replace=False); ys = ytr[idx]
        if len(np.unique(ys)) < 2: continue
        lr = LogisticRegression(penalty='l1', solver='liblinear', C=C, class_weight='balanced', max_iter=500)
        lr.fit(Xtr.values[idx], ys)
        cuenta += (np.abs(lr.coef_[0]) > 1e-8)
    freq = cuenta / n_iter
    return [c for c, f in zip(Xtr.columns, freq) if f >= umbral]

def sel_rf_importance(Xtr, ytr, seed=RANDOM_STATE):
    rf = RandomForestClassifier(n_estimators=300, class_weight='balanced', random_state=seed, n_jobs=-1)
    rf.fit(Xtr, ytr); imp = rf.feature_importances_
    return [c for c, v in zip(Xtr.columns, imp) if v > imp.mean()]

def sel_rfecv(Xtr, ytr, seed=RANDOM_STATE):
    cv = StratifiedKFold(5, shuffle=True, random_state=seed)
    r = RFECV(LogisticRegression(class_weight='balanced', max_iter=1000),
              step=1, cv=cv, scoring='f1', min_features_to_select=1, n_jobs=-1)
    r.fit(Xtr, ytr)
    return [c for c, s in zip(Xtr.columns, r.support_) if s]

def seleccion_2_de_3(*conjuntos):
    cnt = Counter()
    for s in conjuntos: cnt.update(set(s))
    return sorted([f for f, c in cnt.items() if c >= 2])

## 3. Funciones nuevas: especificidad, matriz de confusión y evaluación train/test

`especificidad = TN/(TN+FP)`. `evaluar_train_test` ajusta el modelo en train y devuelve, para **train y
test**: F1, precisión, recall, especificidad, ROC-AUC y la matriz de confusión.

In [5]:
def especificidad(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0,1]).ravel()
    return tn / (tn + fp) if (tn + fp) else 0.0

espec_scorer = make_scorer(especificidad)

def cm_txt(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0,1]).ravel()
    return f'TN={tn} FP={fp} FN={fn} TP={tp}'

def _metricas(y, yp, proba):
    auc = round(roc_auc_score(y, proba), 4) if proba is not None else np.nan
    return {'F1': round(f1_score(y, yp, zero_division=0), 4),
            'Precision': round(precision_score(y, yp, zero_division=0), 4),
            'Recall': round(recall_score(y, yp, zero_division=0), 4),
            'Especificidad': round(especificidad(y, yp), 4),
            'ROC-AUC': auc, 'CM': cm_txt(y, yp)}

def evaluar_train_test(modelo, Xtr, ytr, Xte, yte):
    modelo.fit(Xtr, ytr)
    res = {}
    for split, (X, y) in [('train', (Xtr, ytr)), ('test', (Xte, yte))]:
        try:    proba = modelo.predict_proba(X)[:, 1]
        except Exception: proba = None
        res[split] = _metricas(y, modelo.predict(X), proba)
    return res

## 4. Puntos 1 y 2 — Tabla de clasificadores con **especificidad** y **matriz de confusión (train y test)**

Misma comparación de los 6 clasificadores de `reunion16` (GridSearch en train, evaluación final), pero la
tabla incorpora ahora la **especificidad** y las **matrices de confusión de train y de test**.

In [6]:
def comparar_clasificadores_v2(Xtr, ytr, Xte, yte):
    pos = int(ytr.sum()); neg = len(ytr) - pos; spw = neg / max(pos, 1)
    cv = StratifiedKFold(5, shuffle=True, random_state=RANDOM_STATE)
    grids = {
        'SVM': (SVC(class_weight='balanced', probability=True, random_state=RANDOM_STATE),
                {'C':[0.1,1,10], 'kernel':['linear','rbf'], 'gamma':['scale']}),
        'Random Forest': (RandomForestClassifier(class_weight='balanced', random_state=RANDOM_STATE, n_jobs=-1),
                {'n_estimators':[200], 'max_depth':[None,5,10], 'min_samples_split':[2,5]}),
        'Logistic Regression': (LogisticRegression(class_weight='balanced', max_iter=2000, random_state=RANDOM_STATE),
                {'C':[0.1,1,10], 'solver':['lbfgs']}),
        'Decision Tree': (DecisionTreeClassifier(class_weight='balanced', random_state=RANDOM_STATE),
                {'max_depth':[3,5,None], 'min_samples_split':[2,10]}),
        'KNN': (KNeighborsClassifier(weights='distance'),
                {'n_neighbors':[5,7,11], 'metric':['euclidean','manhattan']}),
        'XGBoost': (XGBClassifier(scale_pos_weight=spw, eval_metric='logloss', random_state=RANDOM_STATE),
                {'n_estimators':[100,200], 'max_depth':[3,5], 'learning_rate':[0.05,0.1]}),
    }
    filas = []
    for nombre, (modelo, params) in grids.items():
        gs = GridSearchCV(modelo, params, cv=cv, scoring='f1', n_jobs=-1)
        gs.fit(Xtr, ytr)
        r = evaluar_train_test(gs.best_estimator_, Xtr, ytr, Xte, yte)
        filas.append({'Modelo': nombre,
                      'F1 (test)': r['test']['F1'], 'Precision (test)': r['test']['Precision'],
                      'Recall (test)': r['test']['Recall'], 'Especificidad (test)': r['test']['Especificidad'],
                      'ROC-AUC (test)': r['test']['ROC-AUC'],
                      'CM train': r['train']['CM'], 'CM test': r['test']['CM']})
    return pd.DataFrame(filas).sort_values('F1 (test)', ascending=False).reset_index(drop=True)

In [7]:
pd.set_option('display.max_colwidth', None); pd.set_option('display.width', 200)
R = {}
for nombre, cols in DATASETS.items():
    print('='*86); print('DATASET', nombre); print('='*86)
    Xtr, Xte, ytr, yte = preparar(coh, cols, NOMINALES)
    s1 = sel_lasso_stability(Xtr, ytr); s2 = sel_rf_importance(Xtr, ytr); s3 = sel_rfecv(Xtr, ytr)
    final = seleccion_2_de_3(s1, s2, s3)
    print(f'Variables seleccionadas (>=2/3) ({len(final)}): {final}')
    tabla = comparar_clasificadores_v2(Xtr[final], ytr, Xte[final], yte)
    cols_show = ['Modelo','F1 (test)','Precision (test)','Recall (test)','Especificidad (test)',
                 'ROC-AUC (test)','CM train','CM test']
    print(tabla[cols_show].to_string(index=False))
    R[nombre] = {'Xtr': Xtr, 'Xte': Xte, 'ytr': ytr, 'yte': yte, 'final': final, 'tabla_clf': tabla}

DATASET 1_embarazo


Variables seleccionadas (>=2/3) (11): ['aumento_peso_gest', 'edad_materna_gest', 'imc_ini_gest', 'peso_ini_gest', 'peso_rn', 'plgf_MoM', 'ratio_MoM', 'ratio_sflt1_plgf', 'tad_1tri', 'talla', 'tas_1tri']


             Modelo  F1 (test)  Precision (test)  Recall (test)  Especificidad (test)  ROC-AUC (test)                 CM train               CM test
            XGBoost     0.5833            0.5385         0.6364                0.9048          0.8889   TN=243 FP=6 FN=0 TP=46  TN=57 FP=6 FN=4 TP=7
Logistic Regression     0.4500            0.3103         0.8182                0.6825          0.7792 TN=174 FP=75 FN=15 TP=31 TN=43 FP=20 FN=2 TP=9
      Random Forest     0.4348            0.4167         0.4545                0.8889          0.8196   TN=246 FP=3 FN=6 TP=40  TN=56 FP=7 FN=6 TP=5
                SVM     0.4103            0.2857         0.7273                0.6825          0.7633 TN=180 FP=69 FN=17 TP=29 TN=43 FP=20 FN=3 TP=8
      Decision Tree     0.3784            0.2692         0.6364                0.6984          0.6638 TN=193 FP=56 FN=10 TP=36 TN=44 FP=19 FN=4 TP=7
                KNN     0.3158            0.3750         0.2727                0.9206          0.7648   TN

Variables seleccionadas (>=2/3) (9): ['ant_medico_actual', 'dieta', 'edad_actual', 'ejercicio', 'estres', 'imc_actual', 'peso_actual', 'tabaco_No', 'tabaco_Si']


             Modelo  F1 (test)  Precision (test)  Recall (test)  Especificidad (test)  ROC-AUC (test)                 CM train               CM test
Logistic Regression     0.4500            0.3103         0.8182                0.6825          0.8196 TN=181 FP=68 FN=15 TP=31 TN=43 FP=20 FN=2 TP=9
            XGBoost     0.4375            0.3333         0.6364                0.7778          0.7633  TN=224 FP=25 FN=1 TP=45 TN=49 FP=14 FN=4 TP=7
                SVM     0.4091            0.2727         0.8182                0.6190          0.7864 TN=180 FP=69 FN=13 TP=33 TN=39 FP=24 FN=2 TP=9
      Decision Tree     0.3750            0.2857         0.5455                0.7619          0.5707 TN=207 FP=42 FN=16 TP=30 TN=48 FP=15 FN=5 TP=6
      Random Forest     0.3529            0.5000         0.2727                0.9524          0.7706   TN=244 FP=5 FN=5 TP=41  TN=60 FP=3 FN=8 TP=3
                KNN     0.3529            0.5000         0.2727                0.9524          0.7172   TN

Variables seleccionadas (>=2/3) (13): ['ant_medico_actual', 'creatinina', 'edad_actual', 'estres', 'grasa_visceral', 'hdl', 'imc_actual', 'ldl', 'peso_actual', 'ratio_cintura_cadera', 'tabaco_Si', 'trigliceridos', 'urato_acidourico']


             Modelo  F1 (test)  Precision (test)  Recall (test)  Especificidad (test)  ROC-AUC (test)                 CM train               CM test
                SVM     0.4324            0.3077         0.7273                0.7143          0.7706 TN=185 FP=64 FN=16 TP=30 TN=45 FP=18 FN=3 TP=8
Logistic Regression     0.4000            0.2759         0.7273                0.6667          0.7778 TN=188 FP=61 FN=14 TP=32 TN=42 FP=21 FN=3 TP=8
            XGBoost     0.4000            0.4444         0.3636                0.9206          0.8384   TN=249 FP=0 FN=0 TP=46  TN=58 FP=5 FN=7 TP=4
      Random Forest     0.2500            0.4000         0.1818                0.9524          0.8124   TN=244 FP=5 FN=6 TP=40  TN=60 FP=3 FN=9 TP=2
                KNN     0.2353            0.3333         0.1818                0.9365          0.7114   TN=249 FP=0 FN=0 TP=46  TN=59 FP=4 FN=9 TP=2
      Decision Tree     0.2000            0.1379         0.3636                0.6032          0.4553  TN=

Variables seleccionadas (>=2/3) (16): ['aumento_peso_gest', 'creatinina', 'edad_materna_gest', 'ejercicio', 'estres', 'grasa_visceral', 'plgf_MoM', 'ratio_MoM', 'ratio_cintura_cadera', 'ratio_sflt1_plgf', 'tad_1tri', 'talla', 'tas_1tri', 'trigliceridos', 'urato_acidourico', 'valor_plgf']


             Modelo  F1 (test)  Precision (test)  Recall (test)  Especificidad (test)  ROC-AUC (test)                 CM train               CM test
      Random Forest     0.5556            0.7143         0.4545                0.9683          0.8081   TN=246 FP=3 FN=4 TP=42  TN=61 FP=2 FN=6 TP=5
            XGBoost     0.5217            0.5000         0.5455                0.9048          0.8341   TN=241 FP=8 FN=0 TP=46  TN=57 FP=6 FN=5 TP=6
Logistic Regression     0.4390            0.3000         0.8182                0.6667          0.7835 TN=192 FP=57 FN=14 TP=32 TN=42 FP=21 FN=2 TP=9
                SVM     0.4324            0.3077         0.7273                0.7143          0.7561 TN=191 FP=58 FN=12 TP=34 TN=45 FP=18 FN=3 TP=8
      Decision Tree     0.3704            0.3125         0.4545                0.8254          0.6299  TN=194 FP=55 FN=9 TP=37 TN=52 FP=11 FN=6 TP=5
                KNN     0.1538            0.5000         0.0909                0.9841          0.8759   TN

## 5. Punto 3 — Barrido de combinaciones de variables (maximizar F1)

Como en clustering, fijamos un algoritmo (**Regresión Logística balanced**) y barremos las
combinaciones de las variables ya seleccionadas en cada dataset. El **F1 se mide por validación cruzada
en train** (la regla es seleccionar solo con train). El tamaño máximo de combinación se acota a un
presupuesto de combinaciones para que sea abordable.

Para las combinaciones con **F1(cv-train) > 0,55** se muestra la tabla completa: F1, precisión, recall,
ROC, especificidad y **matriz de confusión de train y de test**.

In [8]:
LOGREG_BASE = dict(class_weight='balanced', solver='liblinear', max_iter=2000, random_state=RANDOM_STATE)

def max_k(n, budget=5000):
    tot = 0
    for k in range(1, n + 1):
        tot += comb(n, k)
        if tot > budget: return max(k - 1, 1)
    return n

def barrido(Xtr, ytr, variables, budget=5000, cv_folds=4):
    K = max_k(len(variables), budget)
    cv = StratifiedKFold(cv_folds, shuffle=True, random_state=RANDOM_STATE)
    base = LogisticRegression(C=1, **LOGREG_BASE)
    filas = []
    for k in range(1, K + 1):
        for combo in combinations(variables, k):
            f1 = cross_val_score(base, Xtr[list(combo)], ytr, cv=cv, scoring='f1', n_jobs=1).mean()
            filas.append({'k': k, 'combo': combo, 'F1_cv_train': round(f1, 4)})
    barr = pd.DataFrame(filas).sort_values('F1_cv_train', ascending=False).reset_index(drop=True)
    return barr, K

def detalle_combos(Xtr, ytr, Xte, yte, combos):
    cv = StratifiedKFold(5, shuffle=True, random_state=RANDOM_STATE)
    filas = []
    for combo in combos:
        cols = list(combo)
        # CV honesta en train para el ranking (F1 y especificidad)
        yhat = cross_val_predict(LogisticRegression(C=1, **LOGREG_BASE), Xtr[cols], ytr, cv=cv)
        f1cv = round(f1_score(ytr, yhat, zero_division=0), 4)
        espcv = round(especificidad(ytr, yhat), 4)
        r = evaluar_train_test(LogisticRegression(C=1, **LOGREG_BASE), Xtr[cols], ytr, Xte[cols], yte)
        filas.append({'k': len(cols), 'variables': cols,
                      'F1_cv_train': f1cv, 'Espec_cv_train': espcv,
                      'F1_test': r['test']['F1'], 'Precision_test': r['test']['Precision'],
                      'Recall_test': r['test']['Recall'], 'Espec_test': r['test']['Especificidad'],
                      'ROC_test': r['test']['ROC-AUC'],
                      'CM_train': r['train']['CM'], 'CM_test': r['test']['CM']})
    return pd.DataFrame(filas)

In [9]:
for nombre in R:
    print('='*86); print('DATASET', nombre, '- barrido de combinaciones (modelo: LogReg balanced)'); print('='*86)
    Rd = R[nombre]; variables = Rd['final']
    barr, K = barrido(Rd['Xtr'], Rd['ytr'], variables)
    print(f'Variables={len(variables)} | tamaño máx de combinación={K} | combinaciones probadas={len(barr)}')
    surv = barr[barr['F1_cv_train'] > 0.55].copy()
    print(f'Combinaciones con F1(cv-train) > 0.55: {len(surv)}')
    if len(surv) == 0:
        print('Ninguna supera 0.55; se toman las 10 mejores por F1(cv-train) para no dejar el bloque vacío.')
        surv = barr.head(10).copy()
    surv = surv.head(25)   # acotamos el detalle/GridSearch a las 25 mejores
    det = detalle_combos(Rd['Xtr'], Rd['ytr'], Rd['Xte'], Rd['yte'], list(surv['combo']))
    det = det.sort_values('F1_cv_train', ascending=False).reset_index(drop=True)
    cols_show = ['k','F1_cv_train','Espec_cv_train','F1_test','Precision_test','Recall_test',
                 'Espec_test','ROC_test','CM_train','CM_test']
    print('Top combinaciones (ordenadas por F1 de validación cruzada en train):')
    print(det[cols_show].head(15).to_string(index=False))
    print()
    combo_f1   = det.loc[det['F1_cv_train'].idxmax()]
    combo_spec = det.loc[det['Espec_cv_train'].idxmax()]
    print(f'>> Combinación que MAXIMIZA F1   (cv-train={combo_f1["F1_cv_train"]}): {combo_f1["variables"]}')
    print(f'>> Combinación que MAXIMIZA ESPEC (cv-train={combo_spec["Espec_cv_train"]}): {combo_spec["variables"]}')
    Rd['detalle'] = det
    Rd['combo_f1'] = list(combo_f1['variables'])
    Rd['combo_spec'] = list(combo_spec['variables'])

DATASET 1_embarazo - barrido de combinaciones (modelo: LogReg balanced)


Variables=11 | tamaño máx de combinación=11 | combinaciones probadas=2047
Combinaciones con F1(cv-train) > 0.55: 0
Ninguna supera 0.55; se toman las 10 mejores por F1(cv-train) para no dejar el bloque vacío.
Top combinaciones (ordenadas por F1 de validación cruzada en train):
 k  F1_cv_train  Espec_cv_train  F1_test  Precision_test  Recall_test  Espec_test  ROC_test                 CM_train               CM_test
 4       0.4231          0.6908   0.4103          0.2857       0.7273      0.6825    0.6941 TN=174 FP=75 FN=12 TP=34 TN=43 FP=20 FN=3 TP=8
 6       0.4218          0.7189   0.3429          0.2500       0.5455      0.7143    0.6667 TN=178 FP=71 FN=13 TP=33 TN=45 FP=18 FN=5 TP=6
 3       0.4177          0.6827   0.4103          0.2857       0.7273      0.6825    0.6941 TN=171 FP=78 FN=12 TP=34 TN=43 FP=20 FN=3 TP=8
 5       0.4133          0.7068   0.3784          0.2692       0.6364      0.6984    0.6854 TN=175 FP=74 FN=14 TP=32 TN=44 FP=19 FN=4 TP=7
 5       0.4076          0.6

Variables=9 | tamaño máx de combinación=9 | combinaciones probadas=511
Combinaciones con F1(cv-train) > 0.55: 0
Ninguna supera 0.55; se toman las 10 mejores por F1(cv-train) para no dejar el bloque vacío.
Top combinaciones (ordenadas por F1 de validación cruzada en train):
 k  F1_cv_train  Espec_cv_train  F1_test  Precision_test  Recall_test  Espec_test  ROC_test                 CM_train               CM_test
 7       0.4113          0.7349   0.3810          0.2581       0.7273      0.6349    0.7807 TN=182 FP=67 FN=15 TP=31 TN=40 FP=23 FN=3 TP=8
 5       0.4088          0.7470   0.4211          0.2963       0.7273      0.6984    0.8052 TN=183 FP=66 FN=15 TP=31 TN=44 FP=19 FN=3 TP=8
 6       0.4056          0.7269   0.4737          0.3333       0.8182      0.7143    0.8153 TN=177 FP=72 FN=15 TP=31 TN=45 FP=18 FN=2 TP=9
 5       0.4029          0.7390   0.4500          0.3103       0.8182      0.6825    0.7633 TN=185 FP=64 FN=15 TP=31 TN=43 FP=20 FN=2 TP=9
 3       0.4000          0.7028

Variables=13 | tamaño máx de combinación=6 | combinaciones probadas=4095
Combinaciones con F1(cv-train) > 0.55: 0
Ninguna supera 0.55; se toman las 10 mejores por F1(cv-train) para no dejar el bloque vacío.
Top combinaciones (ordenadas por F1 de validación cruzada en train):
 k  F1_cv_train  Espec_cv_train  F1_test  Precision_test  Recall_test  Espec_test  ROC_test                 CM_train               CM_test
 6       0.4196          0.7309   0.3721          0.2500       0.7273      0.6190    0.7619 TN=184 FP=65 FN=13 TP=33 TN=39 FP=24 FN=3 TP=8
 4       0.4138          0.7229   0.4865          0.3462       0.8182      0.7302    0.8110 TN=186 FP=63 FN=16 TP=30 TN=46 FP=17 FN=2 TP=9
 6       0.4138          0.7229   0.4500          0.3103       0.8182      0.6825    0.8066 TN=175 FP=74 FN=15 TP=31 TN=43 FP=20 FN=2 TP=9
 6       0.4027          0.7068   0.4286          0.2903       0.8182      0.6508    0.8167 TN=173 FP=76 FN=16 TP=30 TN=41 FP=22 FN=2 TP=9
 6       0.4000          0.71

Variables=16 | tamaño máx de combinación=4 | combinaciones probadas=2516
Combinaciones con F1(cv-train) > 0.55: 0
Ninguna supera 0.55; se toman las 10 mejores por F1(cv-train) para no dejar el bloque vacío.
Top combinaciones (ordenadas por F1 de validación cruzada en train):
 k  F1_cv_train  Espec_cv_train  F1_test  Precision_test  Recall_test  Espec_test  ROC_test                 CM_train               CM_test
 4       0.4615          0.7028   0.3684          0.2593       0.6364      0.6825    0.7143 TN=177 FP=72 FN=11 TP=35 TN=43 FP=20 FN=4 TP=7
 3       0.4516          0.7028   0.3684          0.2593       0.6364      0.6825    0.7143 TN=177 FP=72 FN=11 TP=35 TN=43 FP=20 FN=4 TP=7
 4       0.4359          0.6948   0.3889          0.2800       0.6364      0.7143    0.7100 TN=177 FP=72 FN=11 TP=35 TN=45 FP=18 FN=4 TP=7
 4       0.4258          0.6948   0.3684          0.2593       0.6364      0.6825    0.7056 TN=180 FP=69 FN=10 TP=36 TN=43 FP=20 FN=4 TP=7
 4       0.4125          0.67

## 6. Punto 4 — GridSearch sobre las combinaciones con F1 > 0,55

Para las dos combinaciones representativas de cada dataset (la que maximiza F1 y la que maximiza
especificidad) se hace **GridSearch** de la Regresión Logística con dos objetivos: **(a) optimizar F1**
y **(b) optimizar especificidad**.

In [10]:
def grid_logreg(Xtr, ytr, cols, scoring):
    cv = StratifiedKFold(5, shuffle=True, random_state=RANDOM_STATE)
    grid = {'C': [0.01, 0.1, 1, 10], 'penalty': ['l1', 'l2']}
    gs = GridSearchCV(LogisticRegression(**LOGREG_BASE), grid, cv=cv, scoring=scoring, n_jobs=-1)
    gs.fit(Xtr[cols], ytr)
    return gs.best_estimator_, gs.best_params_

for nombre in R:
    Rd = R[nombre]
    Rd['grid'] = {}
    for etiqueta, cols in [('combo_f1', Rd['combo_f1']), ('combo_spec', Rd['combo_spec'])]:
        est_f1,   par_f1   = grid_logreg(Rd['Xtr'], Rd['ytr'], cols, 'f1')
        est_spec, par_spec = grid_logreg(Rd['Xtr'], Rd['ytr'], cols, espec_scorer)
        Rd['grid'][etiqueta] = {'cols': cols,
                                'gs_f1': (est_f1, par_f1), 'gs_spec': (est_spec, par_spec)}
    print(f'{nombre}: GridSearch hecho.')
    print(f'   combo max-F1  -> opt.F1 {Rd["grid"]["combo_f1"]["gs_f1"][1]} | opt.Espec {Rd["grid"]["combo_f1"]["gs_spec"][1]}')
    print(f'   combo max-Esp -> opt.F1 {Rd["grid"]["combo_spec"]["gs_f1"][1]} | opt.Espec {Rd["grid"]["combo_spec"]["gs_spec"][1]}')

1_embarazo: GridSearch hecho.
   combo max-F1  -> opt.F1 {'C': 10, 'penalty': 'l2'} | opt.Espec {'C': 0.01, 'penalty': 'l1'}
   combo max-Esp -> opt.F1 {'C': 1, 'penalty': 'l1'} | opt.Espec {'C': 0.01, 'penalty': 'l1'}
2_clasicas: GridSearch hecho.
   combo max-F1  -> opt.F1 {'C': 10, 'penalty': 'l2'} | opt.Espec {'C': 0.01, 'penalty': 'l1'}
   combo max-Esp -> opt.F1 {'C': 10, 'penalty': 'l1'} | opt.Espec {'C': 0.01, 'penalty': 'l1'}


3_clasicas_analitica: GridSearch hecho.
   combo max-F1  -> opt.F1 {'C': 1, 'penalty': 'l1'} | opt.Espec {'C': 0.01, 'penalty': 'l1'}
   combo max-Esp -> opt.F1 {'C': 1, 'penalty': 'l1'} | opt.Espec {'C': 0.01, 'penalty': 'l1'}
4_completo: GridSearch hecho.
   combo max-F1  -> opt.F1 {'C': 10, 'penalty': 'l1'} | opt.Espec {'C': 0.01, 'penalty': 'l1'}
   combo max-Esp -> opt.F1 {'C': 1, 'penalty': 'l2'} | opt.Espec {'C': 0.01, 'penalty': 'l1'}


## 7. Punto 5 — Tablas comparativas (sin GridSearch vs con GridSearch)

Para cada dataset se generan **4 tablas**:
- **Combinación que maximiza F1**: (A1) sin GridSearch vs GridSearch(F1) · (A2) sin GridSearch vs GridSearch(especificidad).
- **Combinación que maximiza especificidad**: (B1) sin GridSearch vs GridSearch(F1) · (B2) sin GridSearch vs GridSearch(especificidad).

"Sin GridSearch" = LogReg base (C=1) del punto 3. Se muestran métricas de test y, además, F1/especificidad
de train y las matrices de confusión de train y test.

In [11]:
def fila_cfg(etiqueta, modelo, Xtr, ytr, Xte, yte, cols):
    r = evaluar_train_test(modelo, Xtr[cols], ytr, Xte[cols], yte)
    return {'Configuración': etiqueta,
            'F1 test': r['test']['F1'], 'Precision test': r['test']['Precision'],
            'Recall test': r['test']['Recall'], 'Espec test': r['test']['Especificidad'],
            'ROC test': r['test']['ROC-AUC'],
            'F1 train': r['train']['F1'], 'Espec train': r['train']['Especificidad'],
            'CM train': r['train']['CM'], 'CM test': r['test']['CM']}

def tablas_comparativas(Rd, etiqueta_combo):
    g = Rd['grid'][etiqueta_combo]; cols = g['cols']
    base = LogisticRegression(C=1, **LOGREG_BASE)
    r_base = fila_cfg('Sin GridSearch (C=1)', base, Rd['Xtr'], Rd['ytr'], Rd['Xte'], Rd['yte'], cols)
    r_gf1  = fila_cfg('GridSearch (optimiza F1)',  g['gs_f1'][0],  Rd['Xtr'], Rd['ytr'], Rd['Xte'], Rd['yte'], cols)
    r_gsp  = fila_cfg('GridSearch (optimiza Espec)', g['gs_spec'][0], Rd['Xtr'], Rd['ytr'], Rd['Xte'], Rd['yte'], cols)
    t1 = pd.DataFrame([r_base, r_gf1])   # punto3 vs GridSearch(F1)
    t2 = pd.DataFrame([r_base, r_gsp])   # punto3 vs GridSearch(Espec)
    return cols, t1, t2

for nombre in R:
    Rd = R[nombre]
    print('#'*86); print('DATASET', nombre); print('#'*86)
    cols_f1, A1, A2 = tablas_comparativas(Rd, 'combo_f1')
    print(f'COMBINACIÓN QUE MAXIMIZA F1: {cols_f1}')
    print('Tabla A1 - sin GridSearch vs GridSearch(F1):')
    print(A1.to_string(index=False)); print()
    print('Tabla A2 - sin GridSearch vs GridSearch(Especificidad):')
    print(A2.to_string(index=False)); print()
    cols_sp, B1, B2 = tablas_comparativas(Rd, 'combo_spec')
    print(f'COMBINACIÓN QUE MAXIMIZA ESPECIFICIDAD: {cols_sp}')
    print('Tabla B1 - sin GridSearch vs GridSearch(F1):')
    print(B1.to_string(index=False)); print()
    print('Tabla B2 - sin GridSearch vs GridSearch(Especificidad):')
    print(B2.to_string(index=False)); print()

######################################################################################
DATASET 1_embarazo
######################################################################################
COMBINACIÓN QUE MAXIMIZA F1: ['imc_ini_gest', 'peso_ini_gest', 'peso_rn', 'talla']
Tabla A1 - sin GridSearch vs GridSearch(F1):
           Configuración  F1 test  Precision test  Recall test  Espec test  ROC test  F1 train  Espec train                 CM train               CM test
    Sin GridSearch (C=1)   0.4103          0.2857       0.7273      0.6825    0.6941    0.4387       0.6988 TN=174 FP=75 FN=12 TP=34 TN=43 FP=20 FN=3 TP=8
GridSearch (optimiza F1)   0.4103          0.2857       0.7273      0.6825    0.6926    0.4387       0.6988 TN=174 FP=75 FN=12 TP=34 TN=43 FP=20 FN=3 TP=8

Tabla A2 - sin GridSearch vs GridSearch(Especificidad):
              Configuración  F1 test  Precision test  Recall test  Espec test  ROC test  F1 train  Espec train                 CM train               CM test

COMBINACIÓN QUE MAXIMIZA ESPECIFICIDAD: ['ant_medico_actual', 'edad_actual', 'imc_actual', 'peso_actual', 'ratio_cintura_cadera', 'tabaco_Si']
Tabla B1 - sin GridSearch vs GridSearch(F1):
           Configuración  F1 test  Precision test  Recall test  Espec test  ROC test  F1 train  Espec train                 CM train               CM test
    Sin GridSearch (C=1)   0.3721            0.25       0.7273       0.619    0.7619    0.4583        0.739 TN=184 FP=65 FN=13 TP=33 TN=39 FP=24 FN=3 TP=8
GridSearch (optimiza F1)   0.3721            0.25       0.7273       0.619    0.7648    0.4615        0.743 TN=185 FP=64 FN=13 TP=33 TN=39 FP=24 FN=3 TP=8

Tabla B2 - sin GridSearch vs GridSearch(Especificidad):
              Configuración  F1 test  Precision test  Recall test  Espec test  ROC test  F1 train  Espec train                 CM train               CM test
       Sin GridSearch (C=1)   0.3721            0.25       0.7273       0.619    0.7619    0.4583        0.739 TN=184 FP=65 FN=13 TP

## 8. Resumen

- **Puntos 1-2**: la tabla de clasificadores de cada dataset ahora incluye **especificidad** y las
  **matrices de confusión de train y test**.
- **Punto 3**: barrido de combinaciones (LogReg balanced) sobre las variables seleccionadas; las que
  superan F1(cv-train) > 0,55 se muestran con todas las métricas.
- **Puntos 4-5**: GridSearch (F1 y especificidad) sobre las combinaciones representativas y 4 tablas
  comparativas por dataset.

> **Lectura clínica:** la especificidad alta con recall bajo significa "pocos falsos positivos pero se
> escapan casos"; en cribado de riesgo suele preferirse **recall/sensibilidad**. Optimizar especificidad
> a secas tiende al clasificador trivial. Las tablas permiten elegir el compromiso según el uso.

In [12]:
print('VARIABLES SELECCIONADAS Y MEJOR COMBINACIÓN POR DATASET')
for nombre, Rd in R.items():
    print('-'*86)
    print(f'{nombre}: seleccionadas={len(Rd["final"])}')
    mejor = Rd['detalle'].iloc[0]
    print(f'   mejor combinación por F1(cv-train)={mejor["F1_cv_train"]} | F1(test)={mejor["F1_test"]} | '
          f'Espec(test)={mejor["Espec_test"]} | vars={mejor["variables"]}')

VARIABLES SELECCIONADAS Y MEJOR COMBINACIÓN POR DATASET
--------------------------------------------------------------------------------------
1_embarazo: seleccionadas=11
   mejor combinación por F1(cv-train)=0.4231 | F1(test)=0.4103 | Espec(test)=0.6825 | vars=['imc_ini_gest', 'peso_ini_gest', 'peso_rn', 'talla']
--------------------------------------------------------------------------------------
2_clasicas: seleccionadas=9
   mejor combinación por F1(cv-train)=0.4113 | F1(test)=0.381 | Espec(test)=0.6349 | vars=['ant_medico_actual', 'dieta', 'edad_actual', 'imc_actual', 'peso_actual', 'tabaco_No', 'tabaco_Si']
--------------------------------------------------------------------------------------
3_clasicas_analitica: seleccionadas=13
   mejor combinación por F1(cv-train)=0.4196 | F1(test)=0.3721 | Espec(test)=0.619 | vars=['ant_medico_actual', 'edad_actual', 'imc_actual', 'peso_actual', 'ratio_cintura_cadera', 'tabaco_Si']
----------------------------------------------------------